<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP9.2A/MP9.2A_soec_steam_channel.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP9.2A · The SOEC Steam Channel

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'channel.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-hydrogen-channel/")
PROJECT = (['mp92a.py', 'MP9.2A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP9.2A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex09.2-hydrogen-channel", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex09.2-hydrogen-channel", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'channel', 'mp92a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import channel as ch
import mp92a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP9.2A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

A solid oxide **electrolysis** cell (SOEC) is the fuel cell of Ex09.2 run backwards: electricity splits steam into hydrogen at the fuel electrode, and the oxide ions cross the electrolyte to the air side. Steam is fed along the same 1 mm channel, steam is taken at the electrode and hydrogen returned, one molecule for one. The operator's main setting is **how fast the steam is fed**. Fed slowly, most of it is converted, but the electrode near the outlet starves of steam and the current there collapses; fed fast, the electrode sees plenty of steam, but most of it leaves unconverted and must be condensed, reheated and fed again. A stack is therefore run at a **target conversion**, typically 60 to 80 %.

So two quantities are wanted: **the steam fraction at the electrode along the channel**, which says where the electrode starves, and **the feed speed that converts 70 % of the steam fed**. A plant controller or a stack optimisation asks them again for every new operating point, so the model is run many times. That is where a fast model pays.

This mini project computes both for any feed speed between 0.2 and 1.2 m/s, with one network.

## A2 · The channel, and what is new against Ex09.2

The channel is Ex09.2's: 100 mm long, 1 mm high, at 800 °C and 1 atm, the gas at Poiseuille's profile from the inlet. The cell below prints its data and draws both directions of the cell.

**Two changes.** First, the cell runs as an electrolyser: **90 % steam and 10 % hydrogen are fed**, and at the electrode steam is taken and hydrogen returned. The exchange is still one for one, so the molar concentration and the velocity are unchanged, exactly as in Ex09.2; only the sign of the wall flux changes.

Second, **the current is no longer given.** Ex09.2 prescribed a uniform 0.5 A/cm², and with it the wall flux $N = i/2F$ was a constant. An electrolyser is run at a fixed cell voltage, and where the steam at the electrode is depleted the concentration overpotential rises and the local current falls. This mini project uses a reduced model of that dependence, a saturating (Langmuir-type) form:

$$\frac{N}{N_0} = g(y_w) = (1 - B) + B\,\frac{r(y_w)}{r(y_\mathrm{in})}, \qquad r(y) = \frac{y}{y + K}$$

where $y_w$ is the steam fraction at the electrode, $N_0 = i_0/2F$ the flux at the inlet's composition with Ex09.2's $i_0$ = 0.5 A/cm², $K$ = 0.2 the steam fraction at which $r$ is half its maximum, and $B$ the **strength of the change**. At $B = 0$ the current is uniform and the problem is Ex09.2's with the flux reversed; this project uses $B = 1$. The form and $K$ are **assumed**: no kinetic constant is taken from a measurement, and a real cell's dependence would be fitted to its polarisation curves.

In [ ]:
# The channel of Ex09.2, run as an electrolyser, and its data
fig = ch.draw_cell()                                     # Ex09.2's drawing: the fuel cell (left) and the electrolyser (right)
plt.show()

rows = [("channel", f"{ch.L_CHANNEL*1e3:.0f} mm long, {ch.H_CHANNEL*1e3:.0f} mm high, {ch.T_CELL-273.15:.0f} °C, 1 atm, c = {ch.C_TOT:.2f} mol/m³"),
        ("gas fed", f"{100*mp.Y_IN:.0f} % steam, {100*(1-mp.Y_IN):.0f} % hydrogen; D = {ch.D_GAS:.1e} m²/s (estimate)"),
        ("feed speed", f"u = {mp.S_MIN:.1f} to {mp.S_MAX:.1f} m/s (mean over the height), a network input"),
        ("current", f"{ch.I_CELL/1e4:.1f} A/cm² at the inlet's composition, then following the steam at the electrode"),
        ("wall flux", f"N0 = i0/2F = {mp.N0:.4f} mol/(m² s) of steam taken, hydrogen returned"),
        ("coupling", f"K = {mp.K_HALF:.2f} (assumed), strength B = {mp.B_COUPLE:.0f}")]
for k, v in rows:
    print(f"{k:>11}  {v}")

**What you should see.** Ex09.2's cross-section of the cell, the fuel cell on the left and the electrolyser on the right: steam into the fuel electrode, hydrogen back into the channel. And the table: everything is Ex09.2's except the feed, the range of speeds and the coupling of the current to the steam.

In [ ]:
# The current density against the steam at the electrode
y = np.linspace(0.01, 0.9, 200)                          # steam fraction at the electrode
theta_w = (y - mp.Y_IN) / mp.Y                           # the same, as the scaled field θ
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(100 * y, mp.current_density(theta_w), label="B = 1, this project")
ax.plot(100 * y, mp.current_density(theta_w, b=0.0), "--", label="B = 0, uniform (Ex09.2 reversed)")
ax.set_xlabel("steam fraction at the electrode [%]"); ax.set_ylabel("current density [A/cm²]"); ax.legend()
plt.tight_layout(); plt.show()
for yy in (0.9, 0.5, 0.3, 0.1):
    print(f"steam at the electrode {100*yy:3.0f} %: current density {float(mp.current_density((yy - mp.Y_IN) / mp.Y)):.2f} A/cm²")

**What you should see.** The current at 0.50 A/cm² with the inlet's 90 % steam, falling slowly at first and then steeply: 0.44 A/cm² at 50 % steam, 0.37 at 30 %, 0.20 at 10 %, and zero as the steam runs out. That is what keeps the steam fraction positive: the electrode cannot take steam that is not there.

## A3 · The domain, the equation and its conditions

The domain is Ex09.2's: the channel in cross-section, $0 < x < L$ along it and $0 < z < h$ across the gap ($z = 0$ the electrode, $z = h$ the interconnect), the velocity Poiseuille's $u(z) = 6u\,(z/h)(1 - z/h)$ at the feed speed $u$. The unknown is the steam fraction $y(x, z)$, which also depends on $u$.

$$u(z)\,\frac{\partial y}{\partial x} = D\,\frac{\partial^2 y}{\partial z^2} \quad \text{in the channel}$$

| where | condition | why it holds |
|---|---|---|
| inlet ($x = 0$) | $y = 0.90$ | the feed: 90 % steam |
| electrode ($z = 0$) | $D c\,\dfrac{\partial y}{\partial z} = N_0\,g(y)$ | steam diffuses into the electrode at the rate the local current takes it; **nonlinear**, since $g$ depends on $y$ at the wall |
| interconnect ($z = h$) | $\partial y/\partial z = 0$ | an impermeable steel wall |
| outlet ($x = L$) | none | diffusion along the channel is dropped, so the equation marches from the inlet and needs no outlet condition |

The reductions of Ex09.2 hold here too, and the numbers are printed by the cell below. The diffusivity is kept at Ex09.2's estimate for steam in hydrogen (the binary diffusivity does not depend on the composition of the pair), and the viscosity of the steam-rich feed is an estimate: the Mach number is below 2e-03, so the gas is incompressible; the exchange is one for one, so the molar concentration $c$ and the velocity do not change; the Reynolds number is a few at most, so the entrance length is under 2 mm (Ex09.2's low-Reynolds estimate, 2 % of the channel) and the velocity is Poiseuille's from the inlet; and after scaling, diffusion along the channel carries the coefficient $(h/L)^2$ = 1e-04 against the term across, so it is dropped and the equation is **parabolic in $x$**.

In [ ]:
# The numbers behind the reductions, at the slowest and fastest feed
print(f"the feed: ρ = {mp.RHO_FEED:.3f} kg/m³, μ = {mp.MU_FEED:.1e} Pa s (estimate), D = {ch.D_GAS:.1e} m²/s (Ex09.2's estimate)")
for s in (mp.S_MIN, mp.S_MAX):
    print(f"u = {s:.1f} m/s: Re = {mp.reynolds(s):.2f}, Péclet along the channel {ch.PE_L*s:.0f}, across {ch.PE_H*s:.2f}, "
          f"pressure drop {mp.pressure_drop(s):.1f} Pa")
print(f"(h/L)² = {ch.EPS**2:.0e}: the coefficient of the dropped axial diffusion after scaling")

**What you should see.** The steam-rich feed is seven times denser than Ex09.2's hydrogen; Reynolds numbers of 0.93 to 5.59, Péclet numbers along the channel of 25 to 150, pressure drops of 9.6 to 57.6 Pa: a slow, laminar, well-mixed flow in which the gas composition changes along the channel and only slightly across it.

---

## A4 · Why there is no exact solution

Ex09.2's channel has an exact solution because the wall flux is a constant: beyond the entrance the field is linear along the channel plus a fixed quartic across it, $\theta = X/s + G\,\varphi(Z)$. Here the flux depends on the steam at the wall, which falls along the channel, so the flux falls too: the mean is no longer linear in $x$, the profile across the channel changes with $x$, and the wall condition multiplies a function of the unknown into its own slope. Separation of variables fails and no closed form exists.

What remains exact is the **balance of the mean**: integrated over the height, the equation and its wall conditions say that the flow-weighted (mixing-cup) mean $\bar y$ falls at the rate the wall takes steam, $u h c\, d\bar y/dx = -N_0\, g(y_w)$. It cannot be solved alone, because it needs the wall value, not the mean. If the gas were mixed across the channel ($y_w = \bar y$, *plug flow*) it could, and the result is the **hand calculation** of section B2. It misses the profile across the channel, and part C measures by how much.

The reference model and the ground truth are therefore **numerical**: a Crank–Nicolson march along the channel (part C). They are checked on grids each twice as fine, by the balance of the mean, and at $B = 0$, where they must reproduce Ex09.2's exact solution with the flux reversed.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the steam fraction at the electrode along the channel, beyond the first 5 mm | the test speeds 0.25, 0.55 and 0.95 m/s, which you do not train on | within **0.05 percentage point** of the ground truth everywhere |
| the feed speed that converts 70 % of the steam fed | found by a gradient through your network | within **1 %** of the ground truth's 0.3201 m/s |
| the time to answer a new speed online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, over the whole range of speeds. Its online answer - the steam fraction along the electrode and the conversion for a speed it has not seen - is timed against the reference model, which must march the channel again for every new speed. The training time is reported apart, as the offline cost.

**Why 0.05 percentage point.** The steam at the electrode lies about 0.1 percentage point below the mixing-cup mean (Ex09.2 measured 0.106 for its uniform flux). The hand calculation, which assumes the gas is mixed across the channel, therefore misses by about that much. An accuracy of half of it is what makes the profile across the channel worth computing; it is also met by an engineering grid of a few dozen nodes (part C).

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network is a function of three inputs and returns one number:

$$(X,\ Z,\ s)\ \longmapsto\ \theta(X, Z; s)$$

with $X = x/L$ and $Z = z/h$ the scaled position, $s = u/(1\ \mathrm{m/s})$ the feed speed in m/s, and $\theta = (y - y_\mathrm{in})/Y$ the scaled change of the steam fraction from the feed, as in Ex09.2: $Y = N_0 L/(u_\mathrm{ref} h c)$ = 0.228 is the fall of the mean over the channel at 1 m/s with the inlet's current. The speed is an **input**, like the coordinates: one training covers every speed from 0.2 to 1.2 m/s, and a new speed is answered by evaluating the network, not by training again.

In [ ]:
# The scales of Ex09.2, and what they become here
print(f"X = x / L,  L = {ch.L_CHANNEL*1e3:.0f} mm;   Z = z / h,  h = {ch.H_CHANNEL*1e3:.0f} mm;   s = u / (1 m/s), {mp.S_MIN} to {mp.S_MAX}")
print(f"θ = (y - {mp.Y_IN}) / Y,   Y = N0 L / (u_ref h c) = {mp.Y:.3f}")
print(f"G = u_ref h² / (D L) = {mp.G:.4f}: the profile across the channel is of order G θ")
print(f"steam fraction at the outlet, mixed across, uniform current: from {mp.Y_IN - mp.Y/mp.S_MAX:.2f} at 1.2 m/s to "
      f"{mp.Y_IN - mp.Y/mp.S_MIN:.2f} at 0.2 m/s (negative: the uniform current would take steam that is not there)")

**What you should see.** The scales of Ex09.2: $Y$ = 0.228, $G$ = 0.0125. And the reason the coupling matters: with a uniform current the slowest feed would end at -0.24, a negative steam fraction. With the current following the steam that cannot happen; the current falls instead.

## B2 · The physics the network is trained with

In the scaled variables the equation and its conditions read

$$6Z(1-Z)\,G\,s\,\frac{\partial \theta}{\partial X} = \frac{\partial^2 \theta}{\partial Z^2}, \qquad
\theta = 0 \ \text{at}\ X = 0, \qquad
\frac{\partial \theta}{\partial Z} = G\,g(\theta) \ \text{at}\ Z = 0, \qquad
\frac{\partial \theta}{\partial Z} = 0 \ \text{at}\ Z = 1$$

with $g(\theta) = N/N_0$ evaluated at the steam fraction $0.90 + Y\theta$ at the wall. The sign of the electrode condition is Ex09.2's reversed: there $\theta_Z = -G$ (steam made at the wall), here $+G\,g$ (steam taken). Each line becomes a loss term:

| loss term | where it is evaluated | what it enforces |
|---|---|---|
| $\mathcal{L}_\mathrm{PDE}$ | points inside $(X, Z, s)$ | the species equation: carried along, diffused across |
| $\mathcal{L}_\mathrm{inlet}$ | points on $X = 0$ | the feed, 90 % steam |
| $\mathcal{L}_\mathrm{electrode}$ | points on $Z = 0$ | the nonlinear wall condition, with $g$ at the network's own steam fraction |
| $\mathcal{L}_\mathrm{interconnect}$ | points on $Z = 1$ | the impermeable wall |
| $\mathcal{L}_\mathrm{balance}$ | quadrature columns at random $(X, s)$ | the balance of the mean, $s\,d\bar\theta/dX = -g(\theta_w)$ |

As in Ex09.2, every residual is **divided by $G$**: the profile across the channel is of order $G$, and so are the residuals of the equation and the walls, so divided they are of order one.

**The balance of the mean.** Integrated over the height with the flow's weight $6Z(1-Z)$ and both wall conditions, the equation gives $s\,d\bar\theta/dX = -g(\theta(X, 0))$: the mean falls at the rate the wall takes steam. A network that satisfied the equation and the walls exactly would satisfy it too; a trained one does not, and Ex09.2 showed that holding the mean by a loss term makes the network tens of times more exact. Here the term also links the mean to the **wall value**, through $g$.

**What can be built in rather than trained.** Ex09.2 built the hand calculation into the network, $\theta = X/s + G\,\mathcal{N}$, so that the network learned only the profile across the channel. The hand calculation here is the plug-flow mean: the gas mixed across the channel, the current at the mean's steam fraction,

$$s\,\frac{d\theta_p}{dX} = -g(\theta_p), \qquad \theta_p(0) = 0$$

which is separable and is solved for any $X$ and $s$ by a few Newton steps (`mp.plug_flow`, written so that autograd differentiates through it). The cell below draws it against the uniform-current line. Whether and how you build it in is your choice in part D.

In [ ]:
# The hand calculation: the plug-flow mean with the current following it, against the uniform current
X = np.linspace(0, 1, 401)
fig, ax = plt.subplots(figsize=(8, 3.4))
for s, c in ((0.25, "tab:red"), (0.55, "tab:blue"), (0.95, "tab:green")):
    ax.plot(100 * X, 100 * mp.steam(mp.plug_flow(X, s)), color=c, label=f"u = {s:.2f} m/s, current following the steam")
    ax.plot(100 * X, 100 * mp.steam(mp.plug_flow(X, s, b=0.0)), color=c, ls="--", lw=0.8)
ax.set_xlabel("x [mm]"); ax.set_ylabel("steam fraction, mixed across [%]"); ax.set_ylim(-10, 95); ax.legend(fontsize=8)
ax.set_title("dashed: the uniform current of Ex09.2, reversed", fontsize=9)
plt.tight_layout(); plt.show()
for s in (0.25, 0.55, 0.95):
    print(f"u = {s:.2f} m/s: steam at the outlet, mixed across: {100*mp.steam(mp.plug_flow(1.0, s)):.2f} % with the current following it, "
          f"{100*mp.steam(mp.plug_flow(1.0, s, b=0.0)):.2f} % with the uniform current")

**What you should see.** Straight dashed lines, the uniform current, and below each a curve that bends: the current falls as the steam is used, so the steam falls more slowly. At 0.25 m/s the uniform current would end at -1.26 % steam, an impossible value; with the current following the steam the outlet sees 14.70 %. At 0.55 m/s the two differ by 2.23 percentage points, at 0.95 m/s by 0.67. This plug-flow mean is not yet the answer: the electrode sees about 0.1 point less steam than the mean, which part C measures.

## B3 · The training points

A physics-informed network needs no solution to train on, only **points** at which the residuals are evaluated. `mp92a.py` provides two samplers, Ex09.2's on this range of speeds:

* `mp.sample_channel(n, seed)`: $n$ points $(X, Z, s)$ inside, a Latin hypercube, with a fifth of them in the first 5 % of the length, where the profile across the channel develops from the flat inlet;
* `mp.sample_faces(n, seed)`: $n$ points on each of the inlet, the electrode and the interconnect, with the same concentration near the inlet on the two walls.

The speed is drawn uniformly from 0.2 to 1.2 m/s. The three **test speeds, 0.25, 0.55 and 0.95 m/s**, are inside the range, but no point is placed at them on purpose: the network is scored where it was not fitted.

In [ ]:
# The training points
pts = mp.sample_channel(4000)                            # (X, Z, s) inside
inlet, electrode_pts, interconnect = mp.sample_faces(400)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].scatter(100 * pts[:, 0], pts[:, 1], s=1, alpha=0.4, label="inside")
ax[0].scatter(100 * electrode_pts[:, 0], electrode_pts[:, 1], s=3, label="electrode")
ax[0].scatter(100 * interconnect[:, 0], interconnect[:, 1], s=3, label="interconnect")
ax[0].scatter(100 * inlet[:, 0], inlet[:, 1], s=3, label="inlet")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("Z = z/h"); ax[0].legend(fontsize=7, loc="center right")
ax[1].hist(pts[:, 2], bins=40)
for s in mp.S_TEST:
    ax[1].axvline(s, color="red", lw=1)
ax[1].set_xlabel("feed speed s [m/s]"); ax[1].set_ylabel("points")
plt.tight_layout(); plt.show()

**What you should see.** Left, the channel (drawn in scaled height) with its points, denser in the first 5 mm; right, a flat distribution of speeds with the three test speeds in red.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the solver an engineer would run: the march on an engineering grid | the same scheme on a grid fine enough to be a hundred times more exact than anything scored |
| grid | 21 × 3 nodes (along × across) | 1601 × 81 nodes |
| its job | to be **timed** against the network: the online cost of answering a new speed, and of finding the speed for 70 % | to **score** the network and the reference |

Both march the same discrete equations, written in `mp92a.March`. This part explains the scheme, builds confidence in the ground truth, and measures the reference model's accuracy and time - the bar part D has to clear.

## C2 · The scheme, step by step

1. **Nodes.** A grid of $n_x \times n_z$ nodes covers the channel, walls included.
2. **Across the channel.** At every node the second derivative across is the central difference $(\theta_{j+1} - 2\theta_j + \theta_{j-1})/h_Z^2$.
3. **The walls by ghost nodes.** The interconnect's ghost mirrors its neighbour (zero slope); the electrode's carries the flux $G\,g(\theta_0)$, which depends on the electrode node's own value.
4. **Along the channel: Crank–Nicolson.** $X$ is stepped like time: each step averages the right-hand side over the old and the new column, which is second order. It needs one sparse solve per step with a matrix factorised once per speed.
5. **The nonlinear electrode: Newton's method on one node.** Only the electrode row depends nonlinearly on the unknown, so each step linearises $g$ about the last estimate and solves again; the rank-one change of the matrix is handled without refactorising (Sherman–Morrison). Three Newton steps reach rounding.
6. **The start: Rannacher's.** At the walls the velocity is zero, so a wall row has no marching term and Crank–Nicolson would hold the wall condition only as the average of two columns. The inlet column breaks it, and Ex09.2 found the error then flips sign at every step and never decays. The first two columns are therefore taken as two fully implicit half steps each.
7. **The flow carried exactly.** The trapezoid rule on $n_z$ nodes integrates $6Z(1-Z)$ to $1 - h_Z^2$, not to 1: on 3 nodes the grid would carry a quarter less gas than is fed and use the steam up too fast. The velocity row is divided by that integral, so the discrete flow is exactly the flow fed, and the balance of the mean holds on any grid. Without it the 21 × 3 grid is 14 percentage points off at 0.25 m/s; with it, 0.04 (`March` in `mp92a.py`).

`mp.March(nx, nz).run(s)` returns the scaled field $\theta$ (nz, nx); `mp.reference(s)` runs it on the engineering grid and returns the electrode's steam fraction and the conversion; `mp.speed_for_conversion(0.70)` finds the speed by the secant method, one march per step.

In [ ]:
# The ground truth: its convergence study at 0.55 m/s, from the truth file
G = np.load("MP9.2A_truth.npz")
print("grid (nx × nz)   steam at the electrode at the outlet   conversion   balance of the mean off by")
for g, yw, c, b in zip(G["conv_grids"], G["conv_y_wall_outlet"], G["conv_conversion"], G["conv_balance"]):
    print(f"{int(g[0]):5d} × {int(g[1]):3d}          {100*yw:.5f} %                    {100*c:.5f} %     {abs(b):.1e}")
d = np.abs(np.diff(G["conv_y_wall_outlet"]))
print("change of the outlet value between grids:", " ".join(f"{v:.1e}" for v in d), "; ratios", np.round(d[:-1] / d[1:], 1))
print(f"Richardson extrapolation: {100*float(G['conv_y_wall_outlet_extrapolated']):.5f} % steam at the electrode at the outlet")
print(f"at B = 0, the march against Ex09.2's exact solution reversed: {float(G['zero_strength_error']):.1e} in the steam fraction at most")

**What you should see.** Five grids, each twice as fine along and across. The change of the electrode's steam fraction at the outlet falls by four each time (5.3e-06, 1.3e-06, 3.3e-07, 8.2e-08: ratios of 4.1, 4.0 and 4.0, the scheme's second order). The ground truth's grid, 1601 × 81, is within 1e-07 of the extrapolated value, five thousand times finer than the 0.05 percentage point asked. The balance of the mean closes to 1e-07. At $B = 0$ the same march reproduces Ex09.2's exact solution with the flux reversed to 1.5e-07 beyond the first 5 mm: the code is right where the answer is known.

In [ ]:
# The zero-strength check, run here: B = 0 against Ex09.2's exact solution with the flux reversed
mp.check_zero_strength()

**What you should see.** Differences of 1.5e-07 in the steam fraction at three speeds, beyond the entrance where Ex09.2's exact solution holds: the march's own discretisation error on 1601 × 81 nodes, ten thousand times below the tolerance.

## C3 · The reference model: accuracy and time

The reference grid was chosen as the coarsest that meets the agreed 0.05 percentage point at every speed of the range: 11 × 3 misses at 0.2 m/s (0.14 point), 21 × 3 meets it everywhere (at most 0.036). The cell below scores it against the ground truth at the test speeds and times it, and finds the speed for 70 % conversion with it. It runs on the CPU of the runtime: a march of 21 small solves gains nothing from a GPU, and an engineer would not run it on one.

In [ ]:
# The reference model against the ground truth, at the three test speeds
worst = 0.0
for s in mp.S_TEST:
    r = mp.reference(s)
    truth = np.interp(r["X"], G["X_wall"], G[f"y_wall_{s:.2f}"])        # the ground truth at the reference's nodes
    e = np.abs(r["y_wall"] - truth)[r["X"] >= mp.X_SCORE].max()
    worst = max(worst, e)
    print(f"u = {s:.2f} m/s: steam at the electrode at the outlet {100*r['y_wall'][-1]:.2f} % (truth {100*G[f'y_wall_{s:.2f}'][-1]:.2f} %), "
          f"worst error {100*e:.3f} points; conversion {100*r['conversion']:.2f} % (truth {100*float(G[f'conversion_{s:.2f}']):.2f} %)")
t_ref = middle(lambda: mp.reference(0.40))               # the online cost of a new speed
s70, n_march, _ = mp.speed_for_conversion(0.70)
t_design = middle(lambda: mp.speed_for_conversion(0.70))  # the online cost of the design question
print(f"\nreference model: worst {100*worst:.3f} percentage points, {t_ref*1e3:.2f} ms per new speed")
print(f"speed for 70 % conversion: {s70:.4f} m/s (truth {float(G['speed_70']):.4f}, {100*(s70/float(G['speed_70'])-1):+.3f} %), "
      f"{n_march} marches, {t_design*1e3:.1f} ms")
print(f"the network must answer a new speed in under {t_ref*1e3/10:.2f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "worst_ref": worst, "t_design_s": t_design, "speed_70_ref": s70}   # kept for part E

**What you should see.** At every test speed the reference model's electrode fraction is within 0.04 percentage point of the ground truth, inside the 0.05 agreed, and its conversion within 0.01 point. One march takes a few milliseconds (1.8 to 7.6 ms on the authors' CPU, shared with other jobs). The speed for 70 % conversion is 0.3200 m/s, 0.02 % from the ground truth's 0.3201, in six marches and some tens of milliseconds. **The reference model is cheap**: 21 small linear solves. That sets a hard bar for the network's online time, which part E measures on this same runtime.

In [ ]:
# The ground truth at the three test speeds: the steam along the electrode, and across the channel at the outlet
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for s in mp.S_TEST:
    ax[0].plot(100 * G["X_wall"], 100 * G[f"y_wall_{s:.2f}"], label=f"u = {s:.2f} m/s, electrode")
    ax[0].plot(100 * G["X_wall"], 100 * mp.steam(mp.plug_flow(G["X_wall"], s)), "k:", lw=0.8)
    f = G[f"field_{s:.2f}"][:, -1]
    ax[1].plot(100 * (f - mp.steam(mp.plug_flow(1.0, s))), G["Z"], label=f"u = {s:.2f} m/s")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("steam fraction [%]"); ax[0].legend(fontsize=8)
ax[0].set_title("dotted: the hand calculation (plug flow)", fontsize=9)
ax[1].set_xlabel("steam fraction at the outlet minus the hand calculation [% points]"); ax[1].set_ylabel("Z = z/h"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
for s in mp.S_TEST:
    e = np.abs(G[f"y_wall_{s:.2f}"] - mp.steam(mp.plug_flow(G["X_wall"], s)))[G["X_wall"] >= mp.X_SCORE].max()
    print(f"u = {s:.2f} m/s: the hand calculation misses the electrode by up to {100*e:.3f} percentage points")

**What you should see.** Left, the steam along the electrode falling from 90 % to 14.68, 50.67 and 66.56 % at the outlet for the three speeds, and the hand calculation (dotted) almost on top of it. Right, the profile across the channel at the outlet: lowest at the electrode, where the steam is taken, about 0.1 point below the plug-flow value, and slightly above it at the interconnect. The hand calculation misses the electrode by up to 0.104, 0.105 and 0.105 percentage points: twice the 0.05 asked, so the network has to resolve this profile.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `net(X, Z, s)`:

| argument | meaning | shape |
|---|---|---|
| `X`, `Z` | $x/L$ and $z/h$, in 0 to 1 | `(n, 1)` |
| `s` | the feed speed in m/s, 0.2 to 1.2 | `(n, 1)` |
| returns | the scaled field $\theta = (y - 0.90)/Y$, with $Y$ = 0.228 | `(n, 1)` |

Whatever you compute inside - the hand calculation, features, several sub-networks - stays inside `net`. The loss below and the evaluation at the end only call `net(X, Z, s)`.

## D2 · The loss, given

`losses(net, seed)` draws training points with the samplers of part B (the same points for the same seed, as L-BFGS needs) and returns the four residual terms of section B2's table: the equation inside, the inlet, the electrode and the interconnect, each divided by $G$. `balance_loss(net, seed)` is the balance of the mean at 300 random $(X, s)$, the mean taken by 16-point Gauss–Legendre quadrature across the height with the flow's weight, its slope along $X$ by autograd. Two details:

* **The electrode's $g$ is evaluated at the network's own wall value, and differentiated through.** Unlike MP8.2A's fin coefficient, $g$ is smooth here, and the optimiser is allowed to see how the flux changes with the wall's steam.
* **The steam fraction is clipped at a millionth inside $g$**, so that an untrained network cannot take the logarithm or the ratio of a negative fraction.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a, grad=False):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1).requires_grad_(grad)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

def g_torch(theta_w):
    """g = N/N0 at the electrode's scaled steam θ_w, in torch (mp.g_flux in NumPy)."""
    y = torch.clamp(mp.Y_IN + mp.Y * theta_w, min=1e-6)                  # the steam fraction at the wall, kept positive
    r = y / (y + mp.K_HALF)
    return (1 - mp.B_COUPLE) + mp.B_COUPLE * r / (mp.Y_IN / (mp.Y_IN + mp.K_HALF))

def losses(net, seed=88, n_in=4000, n_face=400):
    # inside: 6 Z (1 - Z) G s θ_X = θ_ZZ, divided by G
    p = mp.sample_channel(n_in, seed)
    X, Z, s = T(p[:, 0], True), T(p[:, 1], True), T(p[:, 2])
    th = net(X, Z, s)
    r_pde = (d(d(th, Z), Z) - 6 * Z * (1 - Z) * mp.G * s * d(th, X)) / mp.G
    inlet, el, ic = mp.sample_faces(n_face, seed)
    # the inlet, X = 0: θ = 0 (90 % steam), divided by G
    r_in = net(T(inlet[:, 0]), T(inlet[:, 1]), T(inlet[:, 2])) / mp.G
    # the electrode, Z = 0: θ_Z = G g(θ), divided by G
    Ze = T(el[:, 1], True)
    the = net(T(el[:, 0]), Ze, T(el[:, 2]))
    r_el = d(the, Ze) / mp.G - g_torch(the)
    # the interconnect, Z = 1: θ_Z = 0, divided by G
    Zc = T(ic[:, 1], True)
    r_ic = d(net(T(ic[:, 0]), Zc, T(ic[:, 2])), Zc) / mp.G
    return {"pde": r_pde.pow(2).mean(), "inlet": r_in.pow(2).mean(),
            "electrode": r_el.pow(2).mean(), "interconnect": r_ic.pow(2).mean()}

# the balance of the mean (section B2): s dθ̄/dX = -g(θ at the electrode), divided by G
_zq, _wq = np.polynomial.legendre.leggauss(16)            # Gauss-Legendre nodes and weights on [-1, 1]
ZQ = T((_zq + 1) / 2)                                    # moved to [0, 1]
WQ = T(_wq / 2 * 6 * ((_zq + 1) / 2) * (1 - (_zq + 1) / 2)).ravel()   # the weights carry the flow's 6 Z (1 - Z); they sum to 1

def balance_loss(net, seed=7, n=300):
    q = mp.sample_channel(n, seed)[:, [0, 2]]             # (X, s) pairs
    X = T(np.repeat(q[:, 0], 16), True)                  # each pair once per quadrature node
    s = T(np.repeat(q[:, 1], 16))
    th = net(X, ZQ.repeat(n, 1), s).reshape(n, 16)
    mean = (th * WQ).sum(1, keepdim=True)                 # θ̄ at each (X, s)
    rate = (d(th.reshape(-1, 1), X).reshape(n, 16) * WQ).sum(1, keepdim=True)   # dθ̄/dX, by autograd
    Xw, sw = T(q[:, 0]), T(q[:, 1])
    wall = net(Xw, torch.zeros_like(Xw), sw)              # θ at the electrode
    return ((sw * rate + g_torch(wall)) / mp.G).pow(2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The three inputs, scaled to order one ($2X - 1$, $2Z - 1$, the speed mapped to $[-1, 1]$), or features built from them;
* **what is built in.** The hand calculation of section B2 (`mp.plug_flow(X, s, lib=torch)` is differentiable), the inlet condition, the scale $G$ of the profile across the channel;
* **its size.** Layers, neurons, activation. Remember why Part 2 uses tanh, not ReLU.

What the problem tells you, from parts A to C: the field is the plug-flow mean, which changes along the channel by up to 75 percentage points, plus a profile across the channel of about 0.1 point. A network asked to learn both at once must resolve one part in a thousand of its own output.

In [ ]:
# TODO 1 - your network. net(X, Z, s) -> θ, every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, X, Z, s):
        ...                                              # return θ, shape (n, 1)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses` and `balance_loss`. Decide the optimiser or optimisers, the number of steps, the learning rate, the weight of each term, how many points and whether they are drawn afresh. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

The course's recipe (Adam, then L-BFGS on one fixed set of points) is a good start. `quick_score()` below gives the worst error of the steam fraction at the electrode at the three test speeds, beyond 5 mm, in percentage points, from the ground truth, in a fraction of a second.

In [ ]:
# A quick score, given: the worst error at the electrode at the test speeds, percentage points
G = np.load("MP9.2A_truth.npz")
def quick_score():
    Xw = G["X_wall"]; keep = Xw >= mp.X_SCORE; worst = 0.0
    for s in mp.S_TEST:
        with torch.no_grad():
            th = net(T(Xw), T(np.zeros_like(Xw)), T(np.full_like(Xw, s))).cpu().numpy().ravel()
        worst = max(worst, np.abs(mp.steam(th) - G[f"y_wall_{s:.2f}"])[keep].max())
    return 100 * worst

In [ ]:
# TODO 2 - your training
history = {"pde": [], "inlet": [], "electrode": [], "interconnect": [], "balance": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst error at the electrode {quick_score():.3f} percentage points")

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the steam along the electrode at one test speed against the ground truth, with its error. Save each figure you want in the report as `MP9.2A_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the steam fraction along the electrode (401 points) at the test speeds 0.25, 0.55 and 0.95 m/s, at 0.15 and 1.30 m/s outside the range, and the conversion at eleven speeds from 0.2 to 1.2 m/s;
* the field at 0.55 m/s on the ground truth's 401 × 41 nodes;
* **the speed for 70 % conversion by Newton's method through the network**: the conversion is $-Y\bar\theta(1, s)/0.90$, the mean at the outlet by the quadrature of `balance_loss`, and its derivative with respect to $s$ comes from autograd; it starts from the uniform-current hand calculation, $Y/(0.7 \times 0.90)$;
* the **online time** of a new speed on this runtime - the steam along the electrode on 101 points and the conversion, from one call of `net` - and the same for 10 and 100 speeds at once, against the reference model run 1, 10 and 100 times; and the time of the Newton search against the reference model's secant search.

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online times.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
Xe = np.linspace(0.0, 1.0, 401)                          # along the electrode
def electrode_profile(speeds, X=Xe):
    # the steam fraction at the electrode (rows: speeds) and the conversion, from one call of the network
    speeds = np.atleast_1d(speeds); n = len(X)
    Xa = np.r_[np.tile(X, len(speeds)), np.ones(16 * len(speeds))]            # the electrode, then the outlet's quadrature column
    Za = np.r_[np.zeros(n * len(speeds)), np.tile(ZQ.cpu().numpy().ravel(), len(speeds))]
    sa = np.r_[np.repeat(speeds, n), np.repeat(speeds, 16)]
    with torch.no_grad():
        th = net(T(Xa), T(Za), T(sa))
    wall = mp.steam(th[: n * len(speeds)].reshape(len(speeds), n).cpu().numpy())
    mean_out = (th[n * len(speeds):].reshape(len(speeds), 16) * WQ).sum(1).cpu().numpy()
    return wall, -mp.Y * mean_out / mp.Y_IN

speeds = {"test": np.array(mp.S_TEST), "outside": np.array([0.15, 1.30]), "sweep": np.linspace(0.2, 1.2, 11)}
out = {}
for k, v in speeds.items():
    out[f"y_wall_{k}"], out[f"conversion_{k}"] = electrode_profile(v)
    out[f"speeds_{k}"] = v
out["X"] = Xe

# the field at 0.55 m/s on the ground truth's nodes
XX, ZZ = np.meshgrid(G["X"], G["Z"])
with torch.no_grad():
    out["field_0.55"] = mp.steam(net(T(XX), T(ZZ), T(np.full(XX.size, 0.55))).reshape(XX.shape).cpu().numpy())

# the speed for 70 % conversion, by Newton's method through the network
def design(target=0.70, steps=4):
    s = torch.tensor([[mp.Y / (target * mp.Y_IN)]], device=DEVICE, requires_grad=True)   # the uniform-current hand calculation
    for _ in range(steps):
        th = net(torch.ones(16, 1, device=DEVICE), ZQ, s.expand(16, 1))                   # the outlet's quadrature column at speed s
        f = -mp.Y * (th.ravel() * WQ).sum() / mp.Y_IN - target                             # conversion minus the target
        df = torch.autograd.grad(f, s)[0]                                                  # d(conversion)/ds, by autograd
        with torch.no_grad():
            s -= f / df                                                                    # Newton's step
    return float(s.detach())
out["speed_70"] = np.array(design())

# online time: the network against the reference model, on this runtime
timing = {}
for k in (1, 10, 100):
    ps = np.linspace(0.21, 1.19, k)
    timing[f"net_{k}"] = middle(lambda: electrode_profile(ps, np.linspace(0, 1, 101)))
    timing[f"ref_{k}"] = middle(lambda: [mp.reference(p) for p in ps], reps=3 if k == 100 else 5)
timing["design_net"] = middle(design)
timing["design_ref"] = REF["t_design_s"]
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP9.2A_net.pt"))   # the trained weights, beside the report
for k in (1, 10, 100):
    print(f"online, {k:3d} new speed(s): network {timing[f'net_{k}']*1e3:7.2f} ms, reference {timing[f'ref_{k}']*1e3:7.2f} ms "
          f"({timing[f'ref_{k}']/timing[f'net_{k}']:.1f} times)")
print(f"speed for 70 % conversion: network {float(out['speed_70']):.4f} m/s in {timing['design_net']*1e3:.1f} ms, "
      f"reference {REF['speed_70_ref']:.4f} m/s in {timing['design_ref']*1e3:.1f} ms; truth {float(G['speed_70']):.4f} m/s")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's times `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP9.2A_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): worst {100*REF['worst_ref']:.3f} percentage points, {REF['t_ref_s']*1e3:.2f} ms per new speed")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the worst error of the steam fraction at the electrode beyond 5 mm at each test speed, in percentage points;
* the error of the speed for 70 % conversion, in per cent of the ground truth's;
* the largest error of the field at 0.55 m/s (percentage points);
* the network's online time for one new speed, the reference model's on the same runtime, and their ratio; the same for the speed search;
* the offline training time.

Then say in one line each whether the three requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>55}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP9.2A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the error along the electrode at the three test speeds; the error of the field at 0.55 m/s; the conversion against the speed (`R["conversion_sweep"]`) with the 70 % found; and the online times of the network and the reference model for 1, 10 and 100 speeds (`R["net_1"]`, `R["ref_10"]` and so on).

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 6 are about your network and its result, 7 to 9 ask you to use it, 10 concludes. The next cell writes `MP9.2A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features or hand calculation built in), layers, neurons and number of weights. Why this design?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training, and the worst error at the electrode with and without the balance term.": """
""",
    "3. Your training: optimiser(s), steps, learning rates, number of points of each kind, and the offline time on the T4 in seconds.": """
""",
    "4. The steam fraction at the electrode at 0.25, 0.55 and 0.95 m/s: give the worst error at each speed in percentage points. Are all within 0.05? Where along the channel is your error largest, and why there?": """
""",
    "5. The speed for 70 % conversion: your network's value, the ground truth's, the error in per cent, and the Newton steps it took. How much lower is it than the uniform-current hand calculation's 0.362 m/s, and why?": """
""",
    "6. The online time to answer a new speed: your network against the reference model on the same runtime, in milliseconds, and the ratio; the same for 10 and 100 speeds at once, and for the search for the 70 % speed. What does each method spend its time on?": """
""",
    "7. At the 70 % speed, how far does the current density fall along the electrode, from inlet to outlet, in A/cm²? And by how many percentage points does the electrode's steam lie below the mixing-cup mean at the outlet?": """
""",
    "8. Outside the training range: your errors at 0.15 and 1.30 m/s against the ground truth (mp.ground_truth). How fast does the error grow outside the range, and why?": """
""",
    "9. Train again with another seed. How much do the worst electrode errors and the speed for 70 % of the two runs differ? Is your result reproducible within the requirements?": """
""",
    "10. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP9.2A — The SOEC Steam Channel", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP9.2A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP9.2A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP9.2A_report.md into MP9.2A_report.pdf, with any figure
# saved as MP9.2A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP9.2A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP9.2A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP9.2A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP9.2A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass